# Лабораторна робота №4
## Тема: Візуалізація даних 2
### Студент групи ФБ-46 Ільченко Владислав

## Хід виконання роботи

* **Пункт 1:** Реалізувати функцію `harmonic_with_noise(amplitude, frequency, phase, noise_mean, noise_covariance, show_noise)`.
* **Пункт 2:** Створити головне вікно з полем для графіка функції (plot).
* **Пункт 3:** Додати слайдери (sliders), які відповідають за амплітуду, частоту гармоніки, а також слайдери для параметрів шуму.
* **Пункт 4:** Додати чекбокс для перемикання відображення шуму на гармоніці (якщо прапорець прибрано – відображати «чисту гармоніку», якщо ні – зашумлену).
* **Пункт 5:** Додати кнопку «Reset», яка відновлює початкові параметри.
* **Пункт 6:** Забезпечити миттєве оновлення графіка функції гармоніки з накладеним шумом згідно з виставленими параметрами.
* **Пункт 7:** Реалізувати умову: якщо змінено параметри гармоніки, але не змінювалися параметри шуму, то шум має залишитись таким як і був. Якщо змінено параметри шуму, змінюватись має лише шум.
* **Пункт 8:** Реалізувати відновлення початкових параметрів після натискання кнопки «Reset».
* **Пункт 9:** Відфільтрувати отриману зашумлену гармоніку за допомогою фільтра з бібліотеки `scipy.signal` та відобразити відфільтровану «чисту» гармоніку на полі графіка для порівняння.
* **Пункт 10:** Залишити інструкції для користувача, які пояснюють, як користуватися програмою.

In [6]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, CheckButtons
from scipy.signal import butter, lfilter

# Виправлення шляху до системних плагінів Qt для коректного рендеру вікна
venv_base = sys.prefix
os.environ['QT_QPA_PLATFORM_PLUGIN_PATH'] = os.path.join(venv_base, 'Lib', 'site-packages', 'PyQt5', 'Qt5', 'plugins')

# Активація графічного вікна системи
%matplotlib qt

# Початкові константи конфігурації
INIT_AMP, INIT_FREQ, INIT_PHASE, INIT_NOISE_MEAN, INIT_NOISE_COV, INIT_CUTOFF = 1.0, 1.0, 0.0, 0.0, 0.1, 5.0
t_grid = np.linspace(0, 10, 1000)

# Реалізація цифрового фільтра низьких частот Баттерворта (scipy.signal)
def butter_lowpass_filter(data, cutoff, fs=100.0, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    if normal_cutoff >= 1.0: normal_cutoff = 0.99
    elif normal_cutoff <= 0.0: normal_cutoff = 0.01
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return lfilter(b, a, data)

# Реалізація базової функції з ТЗ
def harmonic_with_noise(amplitude, frequency, phase, noise_mean, noise_covariance, show_noise=True):
    y_pure = amplitude * np.sin(2 * np.pi * frequency * t_grid + phase)
    std_dev = np.sqrt(noise_covariance) if noise_covariance >= 0 else 0
    noise = np.random.normal(noise_mean, std_dev, size=len(t_grid))
    y_noisy = y_pure + noise if show_noise else y_pure.copy()
    return y_pure, y_noisy, noise

# Первинна генерація сигналів при старті
y_pure_init, y_noisy_init, global_noise = harmonic_with_noise(
    INIT_AMP, INIT_FREQ, INIT_PHASE, INIT_NOISE_MEAN, INIT_NOISE_COV, show_noise=True
)
y_filt_init = butter_lowpass_filter(y_noisy_init, INIT_CUTOFF)

# Створення головного вікна та поля для графіка
fig, ax = plt.subplots(figsize=(9, 7.5))
plt.subplots_adjust(bottom=0.4)

line_noisy, = ax.plot(t_grid, y_noisy_init, color='orange', alpha=0.8, label='Зашумлена гармоніка')
line_pure, = ax.plot(t_grid, y_pure_init, color='blue', lw=2, linestyle='--', label='Чиста гармоніка')
line_filtered, = ax.plot(t_grid, y_filt_init, color='purple', lw=2, label='Відфільтрована гармоніка')

ax.set_xlim(0, 10)
ax.set_ylim(-2.5, 2.5)
ax.set_xlabel('Час (t)')
ax.set_ylabel('Амплітуда (y)')
ax.set_title('Інтерактивна фільтрація та аналіз сигналів', fontsize=11, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True)

# Розміщення слайдерів (віджетів)
ax_amp = plt.axes([0.25, 0.30, 0.55, 0.025])
ax_freq = plt.axes([0.25, 0.26, 0.55, 0.025])
ax_phase = plt.axes([0.25, 0.22, 0.55, 0.025])
ax_mean = plt.axes([0.25, 0.18, 0.55, 0.025])
ax_cov = plt.axes([0.25, 0.14, 0.55, 0.025])
ax_cutoff = plt.axes([0.25, 0.10, 0.55, 0.025])

s_amp = Slider(ax_amp, 'Amplitude', 0.1, 2.0, valinit=INIT_AMP)
s_freq = Slider(ax_freq, 'Frequency', 0.1, 5.0, valinit=INIT_FREQ)
s_phase = Slider(ax_phase, 'Phase', -np.pi, np.pi, valinit=INIT_PHASE)
s_mean = Slider(ax_mean, 'Noise Mean', -1.0, 1.0, valinit=INIT_NOISE_MEAN)
s_cov = Slider(ax_cov, 'Noise Covariance', 0.0, 1.0, valinit=INIT_NOISE_COV)
s_cutoff = Slider(ax_cutoff, 'Cutoff Frequency', 0.5, 15.0, valinit=INIT_CUTOFF)

# Розміщення кнопок та чекбоксів
ax_check = plt.axes([0.70, 0.02, 0.22, 0.06])
check = CheckButtons(ax_check, ['Show Noise', 'Show Filtered'], [True, True])

ax_reset = plt.axes([0.15, 0.02, 0.12, 0.04])
button_reset = Button(ax_reset, 'Reset', color='lightgray', hovercolor='0.95')

# Функція інтерактивного оновлення параметрів
def update(val):
    global global_noise
    
    # Ізоляція шуму: якщо змінено параметри шуму — регенеруємо тільки його
    if val in [s_mean.val, s_cov.val]:
        std_dev = np.sqrt(s_cov.val) if s_cov.val >= 0 else 0
        global_noise = np.random.normal(s_mean.val, std_dev, size=len(t_grid))
        
    # Перераховуємо чисту гармоніку
    y_p_updated = s_amp.val * np.sin(2 * np.pi * s_freq.val * t_grid + s_phase.val)
    
    # Зчитуємо стан чекбоксів
    show_noise_flag = check.get_status()[0]
    show_filtered_flag = check.get_status()[1]
    
    # Накладання або приховування шуму
    y_n_updated = y_p_updated + global_noise if show_noise_flag else y_p_updated.copy()
    y_f_updated = butter_lowpass_filter(y_p_updated + global_noise, s_cutoff.val)
    
    # Оновлення графічних ліній
    line_pure.set_ydata(y_p_updated)
    line_noisy.set_ydata(y_n_updated)
    line_filtered.set_ydata(y_f_updated)
    
    line_noisy.set_visible(show_noise_flag)
    line_filtered.set_visible(show_filtered_flag)
    
    fig.canvas.draw_idle()

# Зв'язування слайдерів та чекбоксів зі слухачем подій
s_amp.on_changed(update)
s_freq.on_changed(update)
s_phase.on_changed(update)
s_mean.on_changed(update)
s_cov.on_changed(update)
s_cutoff.on_changed(update)
check.on_clicked(update)

# Логіка обробки натискання кнопки «Reset»
def reset_button_clicked(event):
    global global_noise
    s_amp.reset()
    s_freq.reset()
    s_phase.reset()
    s_mean.reset()
    s_cov.reset()
    s_cutoff.reset()
    std_dev = np.sqrt(INIT_NOISE_COV)
    global_noise = np.random.normal(INIT_NOISE_MEAN, std_dev, size=len(t_grid))
    update(None)

button_reset.on_clicked(reset_button_clicked)

plt.show()

### Інструкції для користувача щодо роботи з програмою:
1. **Слайдери Amplitude, Frequency, Phase** змінюють параметри пунктирної синьої лінії (чистої гармоніки). При їх переміщенні форма хвиль накладеного помаранчевого шуму залишається незмінною.
2. **Слайдери Noise Mean, Noise Covariance** регенерують параметри випадкового шуму (помаранчевий графік). Зміна цих повзунків не впливає на чисту синусоїду.
3. **Слайдер Cutoff Frequency** налаштовує частоту зрізу цифрового фільтра низьких частот Баттерворта для згладжування фіолетової лінії.
4. **Чекбокси Show Noise та Show Filtered** перемикають режими видимості відповідних кривих на полі графіка.
5. **Кнопка Reset** повністю відновлює всі початкові параметри системи до вихідного стану.